In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# ── W&B Setup — bulletproof for Kaggle kernels ──
import wandb

_wandb_ready = False
try:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb.login(key=_key, relogin=True)
    _wandb_ready = True
    print("W&B: logged in via Kaggle Secrets")
except Exception as e:
    print(f"W&B: Could not login ({e}), switching to offline mode")
    os.environ["WANDB_MODE"] = "offline"
    _wandb_ready = True  # offline mode still works

def safe_wandb_init(**kwargs):
    """Wrapper that never crashes — falls back to offline mode."""
    try:
        return wandb.init(**kwargs)
    except Exception:
        os.environ["WANDB_MODE"] = "offline"
        return wandb.init(**kwargs)

print(f"W&B mode: {os.environ.get('WANDB_MODE', 'online')}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B: logged in via Kaggle Secrets
W&B mode: online


In [3]:
!pip install lightgbm wandb --quiet
!pip install transformers==4.46.3 accelerate==1.1.1
!pip install -U bitsandbytes>=0.46.1 --quiet
!pip install -q peft trl==0.12.0 datasets
!pip install -q sentence-transformers faiss-cpu sentencepiece

import transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.9 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninst

In [4]:
import pandas as pd
import numpy as np
import os, re, warnings
from itertools import combinations
warnings.filterwarnings("ignore")


# ── MAP@3 Evaluation ──
def map_at_3(y_true, y_pred_top3):
    """Compute Mean Average Precision @ 3."""
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data Loading ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Answer distribution:\n{train['answer'].value_counts()}")

LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Advanced Feature Engineering ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix

def tokenize_words(text):
    return set(re.findall(r'\w+', str(text).lower()))

def jaccard_sim(t1, t2):
    s1, s2 = tokenize_words(t1), tokenize_words(t2)
    if not s1 or not s2: return 0.0
    return len(s1 & s2) / len(s1 | s2)

def common_word_count(t1, t2):
    s1, s2 = tokenize_words(t1), tokenize_words(t2)
    return len(s1 & s2)

def char_ngram_overlap(t1, t2, n=3):
    def ngrams(t):
        t = str(t).lower()
        return set(t[i:i+n] for i in range(len(t)-n+1))
    ng1, ng2 = ngrams(t1), ngrams(t2)
    if not ng1 or not ng2: return 0.0
    return len(ng1 & ng2) / max(len(ng1 | ng2), 1)

def build_features(df, shared_vec=None, svd_model=None, fit=True):
    """Build comprehensive features for LightGBM."""
    prompts = df['prompt'].fillna('').values
    opts = {c: df[c].fillna('').values for c in LABELS}
    N = len(df)

    # ── TF-IDF + SVD ──
    all_text = list(prompts)
    for c in LABELS:
        all_text.extend(list(opts[c]))

    if fit:
        shared_vec = TfidfVectorizer(max_features=15000, ngram_range=(1,2),
                                      sublinear_tf=True, min_df=2)
        shared_vec.fit(all_text)
        svd_model = TruncatedSVD(n_components=128, random_state=42)
        svd_model.fit(shared_vec.transform(all_text))

    prompt_tfidf = shared_vec.transform(prompts)
    opt_tfidf = {c: shared_vec.transform(opts[c]) for c in LABELS}
    prompt_svd = svd_model.transform(prompt_tfidf)
    opt_svd = {c: svd_model.transform(opt_tfidf[c]) for c in LABELS}

    feats = []

    # 1. TF-IDF cosine sim: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array(prompt_tfidf.multiply(opt_tfidf[c]).sum(axis=1)).flatten())

    # 2. SVD cosine sim: prompt ↔ each option
    for c in LABELS:
        norms_p = np.linalg.norm(prompt_svd, axis=1, keepdims=True) + 1e-8
        norms_o = np.linalg.norm(opt_svd[c], axis=1, keepdims=True) + 1e-8
        feats.append(np.sum(prompt_svd * opt_svd[c], axis=1) / (norms_p.flatten() * norms_o.flatten()))

    # 3. Jaccard sim: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([jaccard_sim(prompts[i], opts[c][i]) for i in range(N)]))

    # 4. Common word count: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([common_word_count(prompts[i], opts[c][i]) for i in range(N)]))

    # 5. Char n-gram overlap: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([char_ngram_overlap(prompts[i], opts[c][i]) for i in range(N)]))

    # 6. Option-vs-option Jaccard (pairwise) — 10 pairs
    for c1, c2 in combinations(LABELS, 2):
        feats.append(np.array([jaccard_sim(opts[c1][i], opts[c2][i]) for i in range(N)]))

    # 7. Text length features
    feats.append(np.array([len(str(p)) for p in prompts]))  # prompt char len
    feats.append(np.array([len(str(p).split()) for p in prompts]))  # prompt word count
    for c in LABELS:
        feats.append(np.array([len(str(o)) for o in opts[c]]))  # option char len
        feats.append(np.array([len(str(o).split()) for o in opts[c]]))  # option word count

    # 8. Statistical features across options
    opt_lens = np.column_stack([[len(str(o)) for o in opts[c]] for c in LABELS])
    feats.append(opt_lens.mean(axis=1))   # mean option length
    feats.append(opt_lens.std(axis=1))    # std option length
    feats.append(opt_lens.max(axis=1) - opt_lens.min(axis=1))  # range

    # 9. Relative length features (each option length / mean option length)
    mean_lens = opt_lens.mean(axis=1, keepdims=True) + 1e-8
    for i, c in enumerate(LABELS):
        feats.append(opt_lens[:, i] / mean_lens.flatten())

    dense_feats = np.column_stack(feats)
    combined = csr_matrix(dense_feats)

    return combined, shared_vec, svd_model

print("Building features...")
X_train, shared_vec, svd_model = build_features(train, fit=True)
X_test, _, _ = build_features(test, shared_vec, svd_model, fit=False)
y_train = train['answer'].map(label2id).values
print(f"Features: X_train={X_train.shape}, X_test={X_test.shape}")

# ── Cross-Validation with LightGBM ──
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))

run = safe_wandb_init(
    project="smart-mcq-solver",
    name="model1-enhanced-lgbm",
    config={"model": "LightGBM", "features": "TF-IDF+SVD+Overlap", "n_folds": N_FOLDS}
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        max_depth=5, subsample=0.5, colsample_bytree=0.6,
        reg_alpha=1.0, reg_lambda=5.0, min_child_samples=50,
        random_state=42+fold, n_jobs=-1, verbose=-1
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)]
    )

    val_probs = model.predict_proba(X_val)
    oof_probs[val_idx] = val_probs
    test_probs += model.predict_proba(X_test) / N_FOLDS

    val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs]
    val_true = [id2label[y] for y in y_val]
    fold_map3 = map_at_3(val_true, val_top3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold+1, "fold_map3": fold_map3})

oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_train]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 1 (Enhanced LightGBM) OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("lgbm_test_probs.npy", test_probs)
np.save("lgbm_oof_probs.npy", oof_probs)

test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_lgbm.csv", index=False)
print("Model 1 submission saved to submission_lgbm.csv")
print(sub.head(10))

Train: (2000, 8), Test: (500, 7)
Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Building features...
Features: X_train=(2000, 55), X_test=(500, 55)


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_142751-8xkzmbfj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-enhanced-lgbm
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/8xkzmbfj



--- Fold 1/5 ---
Training until validation scores don't improve for 30 rounds
[100]	valid_0's multi_logloss: 0.17393
[200]	valid_0's multi_logloss: 0.0637321
[300]	valid_0's multi_logloss: 0.0440486
[400]	valid_0's multi_logloss: 0.0420828
Early stopping, best iteration is:
[375]	valid_0's multi_logloss: 0.0420828
Fold 1 MAP@3: 1.0000

--- Fold 2/5 ---
Training until validation scores don't improve for 30 rounds
[100]	valid_0's multi_logloss: 0.178828
[200]	valid_0's multi_logloss: 0.069061
[300]	valid_0's multi_logloss: 0.0485203
[400]	valid_0's multi_logloss: 0.0454169
[500]	valid_0's multi_logloss: 0.0446514
Did not meet early stopping. Best iteration is:
[500]	valid_0's multi_logloss: 0.0446514
Fold 2 MAP@3: 1.0000

--- Fold 3/5 ---
Training until validation scores don't improve for 30 rounds
[100]	valid_0's multi_logloss: 0.156155
[200]	valid_0's multi_logloss: 0.0577297
[300]	valid_0's multi_logloss: 0.0407879
Early stopping, best iteration is:
[359]	valid_0's multi_logloss: 0.0

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:         fold ▁▃▅▆█
wandb:    fold_map3 ████▁
wandb: overall_map3 ▁
wandb: 
wandb: Run summary:
wandb:         fold 5
wandb:    fold_map3 0.99875
wandb: overall_map3 0.99975
wandb: 
wandb: 🚀 View run model1-enhanced-lgbm at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/8xkzmbfj
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260623_142751-8xkzmbfj/logs


Model 1 submission saved to submission_lgbm.csv
   ID Prediction
0   1      A D C
1   2      B D E
2   3      B E A
3   4      E C B
4   5      C A D
5   6      D B A
6   7      E B A
7   8      B D A
8   9      C D A
9  10      B C D


In [5]:
import pandas as pd
import numpy as np
import os, gc, re, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import wandb

# ── Reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Tokenizer ──
from collections import Counter

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

# Build vocabulary from train + test
all_texts_flat = list(train['prompt']) + list(test['prompt'])
for c in LABELS:
    all_texts_flat += list(train[c]) + list(test[c])

word_counts = Counter()
for text in all_texts_flat:
    word_counts.update(tokenize(str(text)))

VOCAB_SIZE = 60000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common(VOCAB_SIZE):
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)}")

def encode_text(text, max_len=128):
    tokens = tokenize(str(text))[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids = ids + [0] * (max_len - len(ids))
    return ids

# ── Load GloVe Embeddings ──
GLOVE_DIM = 100
GLOVE_PATH = "/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    GLOVE_PATH = "glove.6B.100d.txt"

embedding_matrix = np.random.normal(0, 0.05, (len(vocab), GLOVE_DIM)).astype(np.float32)
embedding_matrix[0] = 0  # PAD = zeros

if os.path.exists(GLOVE_PATH):
    print("Loading GloVe embeddings...")
    loaded = 0
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                loaded += 1
    print(f"GloVe loaded: {loaded}/{len(vocab)} words covered")
else:
    print("WARNING: GloVe not found, using random embeddings.")
    print("Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.")

# ── Dataset ──
MAX_PROMPT_LEN = 128
MAX_OPTION_LEN = 128

class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = encode_text(row['prompt'], MAX_PROMPT_LEN)
        option_ids = [encode_text(row[c], MAX_OPTION_LEN) for c in LABELS]

        prompt_t = torch.tensor(prompt_ids, dtype=torch.long)
        options_t = torch.tensor(option_ids, dtype=torch.long)

        if self.is_test:
            return prompt_t, options_t
        else:
            label = label2id[row['answer']]
            return prompt_t, options_t, torch.tensor(label, dtype=torch.long)

# ── Improved Text CNN Model (from scratch, with attention) ──
class AttentionPool(nn.Module):
    """Self-attention pooling layer."""
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x):
        # x: (B, L, D)
        w = torch.softmax(self.attn(x), dim=1)  # (B, L, 1)
        return (x * w).sum(dim=1)  # (B, D)

class TextCNNMCQ(nn.Module):
    """
    Custom Text CNN for Multiple Choice QA (built from scratch).
    Architecture:
    - GloVe embeddings + learnable projection
    - Multi-kernel 1D CNN (sizes 2,3,4,5,7)
    - Attention pooling over CNN features
    - Cross-attention between prompt and option
    - FC classifier with residual connection
    """
    def __init__(self, pretrained_embeddings, kernel_sizes=[2, 3, 4, 5, 7],
                 num_filters=96, hidden_dim=192, dropout=0.5):
        super().__init__()
        vocab_size, embed_dim = pretrained_embeddings.shape

        # Embedding (frozen GloVe + trainable projection)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings), requires_grad=False
        )
        self.embed_proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # Multi-kernel CNN — padding='same' ensures all kernels output same length
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding='same')
            for k in kernel_sizes
        ])
        self.conv_bn = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_out = num_filters * len(kernel_sizes)

        # Attention pooling
        self.attn_pool = AttentionPool(cnn_out)

        # Cross-attention: prompt attends to option
        self.cross_attn = nn.MultiheadAttention(cnn_out, num_heads=4, dropout=0.1, batch_first=True)
        self.cross_norm = nn.LayerNorm(cnn_out)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(cnn_out * 3, hidden_dim),  # prompt + option + cross-attn
            nn.GELU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1)
        )

    def encode_text_seq(self, x):
        """x: (B, L) → (B, L', cnn_out) sequence of features"""
        emb = self.embedding(x)
        emb = self.embed_proj(emb)
        emb = emb.permute(0, 2, 1)  # (B, D, L)

        conv_outs = []
        for conv, bn in zip(self.convs, self.conv_bn):
            c = F.gelu(bn(conv(emb)))  # (B, F, L')
            conv_outs.append(c)

        # Concatenate along filter dim
        out = torch.cat(conv_outs, dim=1)  # (B, cnn_out, L')
        return out.permute(0, 2, 1)  # (B, L', cnn_out)

    def encode_text(self, x):
        """x: (B, L) → (B, cnn_out) pooled features"""
        seq = self.encode_text_seq(x)
        return self.attn_pool(seq)

    def forward(self, prompt, options):
        """
        prompt:  (B, prompt_len)
        options: (B, 5, option_len)
        returns: (B, 5) logits
        """
        B = prompt.size(0)
        prompt_seq = self.encode_text_seq(prompt)   # (B, L, D)
        prompt_feat = self.attn_pool(prompt_seq)     # (B, D)

        logits = []
        for i in range(5):
            opt_seq = self.encode_text_seq(options[:, i, :])  # (B, L, D)
            opt_feat = self.attn_pool(opt_seq)                 # (B, D)

            # Cross-attention: prompt attends to option
            cross_out, _ = self.cross_attn(
                prompt_seq, opt_seq, opt_seq
            )
            cross_feat = self.attn_pool(self.cross_norm(cross_out))  # (B, D)

            combined = torch.cat([prompt_feat, opt_feat, cross_feat], dim=1)
            score = self.fc(combined)
            logits.append(score)

        return torch.cat(logits, dim=1)

# ── Training ──
N_FOLDS = 5
EPOCHS = 12
BATCH_SIZE = 32
LR = 5e-4


safe_wandb_init(project="smart-mcq-solver", name="model2-textcnn-attn",
           config={"model": "TextCNN+Attention", "epochs": EPOCHS, "lr": LR,
                   "batch_size": BATCH_SIZE, "filters": 192, "kernels": "2,3,4,5,7"})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = train['answer'].map(label2id).values
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
test_ds = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Label smoothing cross-entropy
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_preds = F.log_softmax(pred, dim=-1)
        nll = -log_preds.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_preds.mean(dim=-1)
        loss = (1 - self.smoothing) * nll + self.smoothing * smooth_loss
        return loss.mean()

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    train_ds = MCQDataset(train.iloc[tr_idx])
    val_ds   = MCQDataset(train.iloc[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TextCNNMCQ(embedding_matrix).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=1e-3
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    criterion = LabelSmoothingCE(smoothing=0.1)

    best_map3 = 0
    patience = 3
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for prompt, options, labels in train_loader:
            prompt, options, labels = prompt.to(DEVICE), options.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(prompt, options)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds_all, val_labels_all = [], []
        with torch.no_grad():
            for prompt, options, labels in val_loader:
                prompt, options = prompt.to(DEVICE), options.to(DEVICE)
                logits = model(prompt, options)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                val_preds_all.append(probs)
                val_labels_all.append(labels.numpy())

        val_preds_all = np.concatenate(val_preds_all)
        val_labels_all = np.concatenate(val_labels_all)

        val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_preds_all]
        val_true = [id2label[y] for y in val_labels_all]
        epoch_map3 = map_at_3(val_true, val_top3)

        avg_loss = train_loss / len(train_loader)
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
        wandb.log({"fold": fold+1, "epoch": epoch+1, "train_loss": avg_loss, "val_map3": epoch_map3})

        if epoch_map3 > best_map3:
            best_map3 = epoch_map3
            best_val_preds = val_preds_all.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"Fold {fold+1} Best MAP@3: {best_map3:.4f}")
    oof_probs[val_idx] = best_val_preds

    # Test inference with best model
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    fold_test_preds = []
    with torch.no_grad():
        for prompt, options in test_loader:
            prompt, options = prompt.to(DEVICE), options.to(DEVICE)
            logits = model(prompt, options)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            fold_test_preds.append(probs)
    test_probs += np.concatenate(fold_test_preds) / N_FOLDS

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 2 (TextCNN+Attn) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("textcnn_test_probs.npy", test_probs)
np.save("textcnn_oof_probs.npy", oof_probs)

test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_textcnn.csv", index=False)
print("Model 2 submission saved to submission_textcnn.csv")
print(sub.head(10))

Device: cuda
Vocabulary size: 2975
Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.


wandb: setting up run pemfbxtn
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_142802-pemfbxtn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-textcnn-attn
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/pemfbxtn



Fold 1/5
Epoch 1/12 | Loss: 1.6192 | Val MAP@3: 0.5204
Epoch 2/12 | Loss: 1.3766 | Val MAP@3: 0.7742
Epoch 4/12 | Loss: 0.6434 | Val MAP@3: 0.9429
Epoch 6/12 | Loss: 0.5093 | Val MAP@3: 0.9717
Epoch 8/12 | Loss: 0.4643 | Val MAP@3: 0.9850
Epoch 10/12 | Loss: 0.4420 | Val MAP@3: 0.9812
  Early stopping at epoch 11
Fold 1 Best MAP@3: 0.9850

Fold 2/5
Epoch 1/12 | Loss: 1.6255 | Val MAP@3: 0.5596
Epoch 2/12 | Loss: 1.4068 | Val MAP@3: 0.8083
Epoch 4/12 | Loss: 0.6774 | Val MAP@3: 0.9621
Epoch 6/12 | Loss: 0.5200 | Val MAP@3: 0.9875
Epoch 8/12 | Loss: 0.4612 | Val MAP@3: 0.9925
Epoch 10/12 | Loss: 0.4412 | Val MAP@3: 0.9925
  Early stopping at epoch 11
Fold 2 Best MAP@3: 0.9925

Fold 3/5
Epoch 1/12 | Loss: 1.6140 | Val MAP@3: 0.5346
Epoch 2/12 | Loss: 1.3766 | Val MAP@3: 0.8246
Epoch 4/12 | Loss: 0.6375 | Val MAP@3: 0.9746
Epoch 6/12 | Loss: 0.5084 | Val MAP@3: 0.9975
Epoch 8/12 | Loss: 0.4537 | Val MAP@3: 0.9975
Epoch 10/12 | Loss: 0.4312 | Val MAP@3: 0.9975
  Early stopping at epoch 10


wandb: updating run metadata



=== Model 2 (TextCNN+Attn) Overall OOF MAP@3: 0.9937 ===


wandb: uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:        epoch ▁▂▂▃▄▅▆▇█▁▄▅▅▆▇█▁▂▂▃▅▅▆▇▇▂▃▄▅▅▇▇█▁▂▄▅▅▆█
wandb:         fold ▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆███████
wandb: overall_map3 ▁
wandb:   train_loss █▇▂▂▁▁▁▁█▇▂▂▂▁▁▁▁▇▃▂▁▁▁▁▁▄▂▂▂▁▁▁▁▇▃▂▂▁▁▁
wandb:     val_map3 ▁▅▇▇███▂▅▇██████▁▅▇█████▂▇▇█████▂▄▆█████
wandb: 
wandb: Run summary:
wandb:        epoch 11
wandb:         fold 5
wandb: overall_map3 0.99367
wandb:   train_loss 0.46072
wandb:     val_map3 0.98833
wandb: 
wandb: 🚀 View run model2-textcnn-attn at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/pemfbxtn
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260623_142802-pemfbxtn/logs


Model 2 submission saved to submission_textcnn.csv
   ID Prediction
0   1      A C B
1   2      B A E
2   3      B A E
3   4      E D A
4   5      C A B
5   6      D E B
6   7      E C B
7   8      B A E
8   9      C E D
9  10      B D E


In [6]:
import os, re, warnings
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import wandb
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']

SIM_PIN = 0.93          # similarity threshold to trust a match
norm = lambda s: ' '.join(str(s).lower().split())

def full_text(df):
    return (df['prompt'].astype(str) + ' ' +
            df[LABELS].astype(str).agg(' '.join, axis=1))

ft_tr, ft_te = full_text(train), full_text(test)
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
vec.fit(pd.concat([ft_tr, ft_te]))
Xtr, Xte = vec.transform(ft_tr), vec.transform(ft_te)

# ── Internal validation: simulate matcher within train itself ──
S_tt = cosine_similarity(Xtr, Xtr)
np.fill_diagonal(S_tt, -1.0)
nn_idx, nn_sim = S_tt.argmax(1), S_tt.max(1)
mapped = correct = 0
for i in range(len(train)):
    if nn_sim[i] >= SIM_PIN:
        j = nn_idx[i]
        ans_text = norm(train.iloc[j][train.iloc[j]['answer']])
        opts = [norm(train.iloc[i][c]) for c in LABELS]
        if ans_text in opts:
            mapped += 1
            if LABELS[opts.index(ans_text)] == train.iloc[i]['answer']:
                correct += 1
precision = correct / max(mapped, 1)
print(f"Internal validation: mapped={mapped}/{len(train)}, precision={precision:.4f}")
del S_tt

# ── Match test → train ──
S = cosine_similarity(Xte, Xtr)            # 500 x 2000
best_idx, best_sim = S.argmax(1), S.max(1)

pinned = np.full(len(test), -1, dtype=int)   # pinned letter index, -1 = none
for i in range(len(test)):
    if best_sim[i] >= SIM_PIN:
        j = best_idx[i]
        ans_text = norm(train.iloc[j][train.iloc[j]['answer']])
        opts = [norm(test.iloc[i][c]) for c in LABELS]
        if ans_text in opts:
            pinned[i] = opts.index(ans_text)

n_pinned = int((pinned >= 0).sum())
print(f"Test rows pinned by duplicate matcher: {n_pinned}/{len(test)} "
      f"({n_pinned/len(test):.1%})")
np.save("dup_pinned.npy", pinned)
np.save("dup_sim.npy", best_sim)
np.save("dup_match_idx.npy", best_idx)

# ── Standalone fallback submission (dup layer + class-prior fill) ──
prior_order = [LABELS.index(l) for l in
               train['answer'].value_counts().index.tolist()]
rows = []
for i in range(len(test)):
    if pinned[i] >= 0:
        rest = [l for l in prior_order if l != pinned[i]]
        top3 = [pinned[i]] + rest[:2]
    else:
        top3 = prior_order[:3]
    rows.append(" ".join(LABELS[k] for k in top3))
sub = pd.DataFrame({"ID": test["id"], "Prediction": rows})
sub.to_csv("submission_dupmatch.csv", index=False)
print("Saved submission_dupmatch.csv (deterministic-layer fallback)")

run = safe_wandb_init(project="smart-mcq-solver", name="stage-a-duplicate-matcher",
                      config={"sim_pin": SIM_PIN, "ngram": "1-2"})
wandb.log({"internal_mapped": mapped, "internal_precision": precision,
           "test_pinned": n_pinned, "test_pin_rate": n_pinned/len(test)})
wandb.finish()

Internal validation: mapped=1743/2000, precision=1.0000
Test rows pinned by duplicate matcher: 475/500 (95.0%)
Saved submission_dupmatch.csv (deterministic-layer fallback)


wandb: setting up run rmbvv2qq
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_143249-rmbvv2qq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stage-a-duplicate-matcher
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/rmbvv2qq
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary; uploading wandb-metadata.json; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:    internal_mapped ▁
wandb: internal_precision ▁
wandb:      test_pin_rate ▁
wandb:        test_pinned ▁
wandb: 
wandb: Run summary:
wandb:    internal_mapped 1743
wandb: internal_precision 1
wandb:      test_pin_rate 0.95
wandb:        test_pinned 

In [7]:
import os, gc, glob, warnings
import numpy as np, pandas as pd
import wandb
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']

TOPK_DENSE, TOPK_FINAL, MAX_CTX_CHARS = 10, 3, 2200

# ── 1. Locate a paragraph corpus ──
CORPUS_DIRS = [
    "/kaggle/input/all-paraphs-parsed-expanded",
    "/kaggle/input/stem-wiki-cohere-no-emb",
    "/kaggle/input/wikipedia-stem-plaintext",
    "/kaggle/input/270k-wikipedia-stem-articles",
]
corpus_texts = None
for d in CORPUS_DIRS:
    if os.path.exists(d):
        files = sorted(glob.glob(f"{d}/**/*.parquet", recursive=True)) or \
                sorted(glob.glob(f"{d}/**/*.csv", recursive=True))
        parts = []
        for f in files[:20]:
            df = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)
            col = next((c for c in ["text", "paragraph", "context", "content"]
                        if c in df.columns), None)
            if col: parts.append(df[col].astype(str))
        if parts:
            corpus_texts = pd.concat(parts, ignore_index=True).drop_duplicates().values
            print(f"Corpus loaded from {d}: {len(corpus_texts):,} paragraphs")
            break

if corpus_texts is None:
    print("WARNING: no Wikipedia corpus attached — proceeding CLOSED-BOOK "
          "(context=''). Attach 'all-paraphs-parsed-expanded' for full score.")
    train["context"] = ""; test["context"] = ""
else:
    # ── 2. Dense retrieval with sentence-transformers + FAISS ──
    from sentence_transformers import SentenceTransformer
    emb_model = None
    for p in ["/kaggle/input/sentence-transformers/all-minilm-l6-v2",
              "/kaggle/input/all-minilm-l6-v2",
              "sentence-transformers/all-MiniLM-L6-v2"]:
        try:
            emb_model = SentenceTransformer(p, device="cuda" if
                __import__("torch").cuda.is_available() else "cpu")
            print(f"Embedding model: {p}"); break
        except Exception:
            continue
    assert emb_model is not None, "No sentence-transformer available — attach model dataset or enable internet"

    corpus_emb = emb_model.encode(list(corpus_texts), batch_size=512,
                                  normalize_embeddings=True,
                                  show_progress_bar=True).astype(np.float32)

    def make_queries(df):
        return (df["prompt"].astype(str) + " " +
                df[LABELS].astype(str).agg(" ".join, axis=1)).tolist()

    q_all = make_queries(train) + make_queries(test)
    q_emb = emb_model.encode(q_all, batch_size=256,
                             normalize_embeddings=True,
                             show_progress_bar=True).astype(np.float32)
    try:
        import faiss
        index = faiss.IndexFlatIP(corpus_emb.shape[1]); index.add(corpus_emb)
        _, I = index.search(q_emb, TOPK_DENSE)
    except Exception:
        from sklearn.neighbors import NearestNeighbors
        nn = NearestNeighbors(n_neighbors=TOPK_DENSE, metric="cosine").fit(corpus_emb)
        _, I = nn.kneighbors(q_emb)

    # ── 3. TF-IDF lexical re-rank of dense candidates ──
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    cand_ids = np.unique(I.ravel())
    sub_texts = {int(c): str(corpus_texts[c]) for c in cand_ids}
    rr_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                             max_features=200000)
    rr_vec.fit(list(sub_texts.values()) + q_all)

    contexts = []
    Q = rr_vec.transform(q_all)
    for qi in range(len(q_all)):
        cands = [int(c) for c in I[qi]]
        Cm = rr_vec.transform([sub_texts[c] for c in cands])
        sims = cosine_similarity(Q[qi], Cm).ravel()
        order = np.argsort(-sims)[:TOPK_FINAL]
        ctx = " ".join(sub_texts[cands[k]] for k in order)
        contexts.append(ctx[:MAX_CTX_CHARS])

    train["context"] = contexts[:len(train)]
    test["context"]  = contexts[len(train):]
    del corpus_emb, q_emb; gc.collect()

train[["id", "context"]].to_csv("train_context.csv", index=False)
test[["id", "context"]].to_csv("test_context.csv", index=False)
print(f"Contexts saved. Avg context chars — train: "
      f"{train['context'].str.len().mean():.0f}, "
      f"test: {test['context'].str.len().mean():.0f}")

run = safe_wandb_init(project="smart-mcq-solver", name="stage-b-rag-retrieval",
                      config={"topk_dense": TOPK_DENSE, "topk_final": TOPK_FINAL,
                              "max_ctx_chars": MAX_CTX_CHARS,
                              "corpus_size": 0 if corpus_texts is None else len(corpus_texts)})
wandb.log({"train_ctx_chars": float(train['context'].str.len().mean()),
           "test_ctx_chars": float(test['context'].str.len().mean())})
wandb.finish()

Contexts saved. Avg context chars — train: 0, test: 0


wandb: setting up run xxlm8kzj
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_143252-xxlm8kzj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stage-b-rag-retrieval
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/xxlm8kzj
wandb: updating run metadata; uploading summary
wandb: updating run metadata
wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json
wandb: uploading config.yaml; uploading wandb-metadata.json; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:  test_ctx_chars ▁
wandb: train_ctx_chars ▁
wandb: 
wandb: Run summary:
wandb:  test_ctx_chars 0
wandb: train_ctx_chars 0
wandb: 
wandb: 🚀 View run stage-b-rag-retrieval at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/xxlm8kzj
wandb: ⭐️ View project 

In [8]:
import os, gc, warnings
import numpy as np, pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
import wandb
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1); break
        scores.append(score)
    return np.mean(scores)

# Merge RAG contexts produced in Stage B (fallback: closed-book)
for df, f in [(train, "train_context.csv"), (test, "test_context.csv")]:
    if os.path.exists(f):
        ctx = pd.read_csv(f)
        df["context"] = ctx["context"].fillna("").values
    else:
        df["context"] = ""

MODEL_CANDS = ["/kaggle/input/deberta-v3-large/pytorch/default/1",
               "/kaggle/input/debertav3large",
               "/kaggle/input/deberta-v3-large",
               "microsoft/deberta-v3-large"]
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
MODEL_PATH = None
for p in MODEL_CANDS:
    if not p.startswith("/") or os.path.exists(p):
        try:
            tokenizer = AutoTokenizer.from_pretrained(p)
            MODEL_PATH = p; break
        except Exception:
            continue
print(f"DeBERTa from: {MODEL_PATH}")

MAX_LEN, N_FOLDS, EPOCHS, FREEZE_LAYERS = 512, 5, 2, 12

def tokenize_block(df):
    enc_ids, enc_mask = [], []
    for _, r in df.iterrows():
        firsts  = [str(r.get("context", ""))] * 5
        seconds = [f"{r['prompt']} [SEP] {r[c]}" for c in LABELS]
        out = tokenizer(firsts, seconds, truncation=True,
                        max_length=MAX_LEN, padding=False)
        enc_ids.append(out["input_ids"]); enc_mask.append(out["attention_mask"])
    return enc_ids, enc_mask

class MCQDataset(torch.utils.data.Dataset):
    def __init__(self, ids, mask, labels=None):
        self.ids, self.mask, self.labels = ids, mask, labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        item = {"input_ids": self.ids[i], "attention_mask": self.mask[i]}
        if self.labels is not None: item["label"] = int(self.labels[i])
        return item

def collate(batch):
    flat = [{"input_ids": ids, "attention_mask": m}
            for b in batch for ids, m in zip(b["input_ids"], b["attention_mask"])]
    padded = tokenizer.pad(flat, return_tensors="pt")
    bs = len(batch)
    out = {k: v.view(bs, 5, -1) for k, v in padded.items()}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return out

print("Tokenizing…")
tr_ids, tr_mask = tokenize_block(train)
te_ids, te_mask = tokenize_block(test)
y_all = train["answer"].map(label2id).values
test_ds = MCQDataset(te_ids, te_mask)

oof_probs  = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

run = safe_wandb_init(project="smart-mcq-solver", name="model3a-deberta-rag",
                      config={"model": MODEL_PATH, "max_len": MAX_LEN,
                              "n_folds": N_FOLDS, "epochs": EPOCHS,
                              "freeze_layers": FREEZE_LAYERS, "lr": 1e-5})

for fold, (tr_idx, val_idx) in enumerate(skf.split(np.zeros(len(train)), y_all)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    model = AutoModelForMultipleChoice.from_pretrained(MODEL_PATH)
    # freeze embeddings + lower encoder layers (stability on 2k samples)
    for p_ in model.deberta.embeddings.parameters(): p_.requires_grad = False
    for layer in model.deberta.encoder.layer[:FREEZE_LAYERS]:
        for p_ in layer.parameters(): p_.requires_grad = False

    args = TrainingArguments(
        output_dir=f"/kaggle/working/deberta_f{fold}",
        num_train_epochs=EPOCHS,
        learning_rate=1e-5,
        warmup_ratio=0.1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        fp16=torch.cuda.is_available(),
        logging_steps=25,
        save_strategy="no",
        report_to="wandb",
        remove_unused_columns=False,
    )
    tr_ds = MCQDataset([tr_ids[i] for i in tr_idx], [tr_mask[i] for i in tr_idx],
                       y_all[tr_idx])
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds,
                      data_collator=collate)
    trainer.train()

    val_ds = MCQDataset([tr_ids[i] for i in val_idx], [tr_mask[i] for i in val_idx])
    val_logits = trainer.predict(val_ds).predictions
    val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
    oof_probs[val_idx] = val_probs
    test_probs += torch.softmax(torch.tensor(
        trainer.predict(test_ds).predictions), dim=1).numpy() / N_FOLDS

    val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs]
    val_true = [id2label[y] for y in y_all[val_idx]]
    fold_map3 = map_at_3(val_true, val_top3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold + 1, "fold_map3": fold_map3})

    del model, trainer; gc.collect(); torch.cuda.empty_cache()

oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 3a (DeBERTa-v3-large + RAG) OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("deberta_oof_probs.npy", oof_probs)
np.save("deberta_test_probs.npy", test_probs)
test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({"ID": test["id"],
                    "Prediction": [" ".join(t) for t in test_top3]})
sub.to_csv("submission_deberta.csv", index=False)
print("Model 3a submission saved to submission_deberta.csv")


2026-06-23 14:32:59.002851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782225179.220178      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782225179.280062      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782225179.859291      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782225179.859335      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782225179.859338      23 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

DeBERTa from: microsoft/deberta-v3-large
Tokenizing…


...

Fold 5 MAP@3: 0.9071


wandb: updating run metadata



=== Model 3a (DeBERTa-v3-large + RAG) OOF MAP@3: 0.9050 ===


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:                    fold ▁▃▅▆█
wandb:               fold_map3 ▁▅██▅
wandb:            overall_map3 ▁
wandb:            test/runtime ▁█▄█▂█▅█▃█
wandb: test/samples_per_second █▅▃▅▅▅▁▅▄▅
wandb:   test/steps_per_second █▆▃▆▅▆▁▆▃▆
wandb:             train/epoch ▁▃▆██▁▃▆██▁▃▆██▁▃▆██▁▃▆██
wandb:       train/global_step ▁▃▆█████▁▃▆█████▁▃▆█████▁▃▆█████▁▃▆█████
wandb:         train/grad_norm ▁▂▃▄▂▂▃▅▂▄▅▆▁▃▄▆▁▂▇█
wandb:     train/learning_rate █▆▃▁█▆▃▁█▆▃▁█▆▃▁█▆▃▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:                    fold 5
wandb:               fold_map3 0.90708
wandb:            overall_map3 0.905
wandb:            test/runtime 41.9523
wandb: test/samples_per_second 11.918
wandb:   test/steps_per_second 0.763
wandb:              total_flos 2593690221966720.0
wandb:             train/epoch 

Model 3a submission saved to submission_deberta.csv


In [9]:
import pandas as pd, numpy as np, os, gc, re, warnings, torch
warnings.filterwarnings("ignore")


# ═══ GPU CHECK — LLM requires GPU, will timeout on CPU ═══
if not torch.cuda.is_available():
    print("=" * 60)
    print("WARNING: NO GPU DETECTED!")
    print("The LLM cell REQUIRES a GPU to finish within time limits.")
    print("Go to: Settings → Accelerator → GPU T4 x2")
    print("=" * 60)
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

LABELS = ['A','B','C','D','E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i:l for i,l in enumerate(LABELS)}
def map_at_3(y_true, y_pred_top3):
    return np.mean([1.0/(next((i+1 for i,p in enumerate(pr[:3]) if p==t),4))
                     for t,pr in zip(y_true,y_pred_top3)])

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

# Merge RAG contexts from Stage B
for df, f in [(train, "train_context.csv"), (test, "test_context.csv")]:
    if os.path.exists(f):
        df["context"] = pd.read_csv(f)["context"].fillna("").values
    else:
        df["context"] = ""

# ═══ LOAD MODEL ═══
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_path = None
for p in ["/kaggle/input/qwen2.5/transformers/7b-instruct/1",
          "/kaggle/input/qwen-2-5/transformers/7b-instruct/1",
          "Qwen/Qwen2.5-7B-Instruct"]:
    if not p.startswith("/") or os.path.exists(p):
        model_path = p
        break
print(f"Loading: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_path,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    ),
    device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
model.eval()
print("✓ Model loaded")


RUN_QLORA = False
if RUN_QLORA:
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        task_type="CAUSAL_LM"))
    model.print_trainable_parameters()

    def build_example(row):
        msgs = [{"role": "user", "content":
                 f"Question: {row['prompt']}\n\n" +
                 "\n".join(f"{c}) {row[c]}" for c in LABELS) +
                 "\n\nAnswer with ONLY the correct letter:"}]
        prompt_text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        ans_ids = tokenizer(str(row["answer"]) + tokenizer.eos_token,
                            add_special_tokens=False)["input_ids"]
        input_ids = (prompt_ids + ans_ids)[-1024:]
        labels = [-100] * (len(input_ids) - len(ans_ids)) + ans_ids
        return {"input_ids": input_ids, "labels": labels,
                "attention_mask": [1] * len(input_ids)}

    examples = [build_example(r) for _, r in train.iterrows()]
    def collate_ft(batch):
        ml = max(len(b["input_ids"]) for b in batch)
        pad = tokenizer.pad_token_id
        return {
            "input_ids": torch.tensor([b["input_ids"] + [pad]*(ml-len(b["input_ids"])) for b in batch]),
            "attention_mask": torch.tensor([b["attention_mask"] + [0]*(ml-len(b["attention_mask"])) for b in batch]),
            "labels": torch.tensor([b["labels"] + [-100]*(ml-len(b["labels"])) for b in batch]),
        }
    from transformers import Trainer, TrainingArguments
    safe_wandb_init(project="smart-mcq-solver", name="model3b-qlora-corrected",
               config={"r": 16, "alpha": 32, "loss": "completion-only", "epochs": 1})
    trainer = Trainer(model=model, train_dataset=examples,
        data_collator=collate_ft,
        args=TrainingArguments(output_dir="/kaggle/working/qlora",
            num_train_epochs=1, per_device_train_batch_size=2,
            gradient_accumulation_steps=8, learning_rate=1e-4,
            lr_scheduler_type="cosine", warmup_ratio=0.1, fp16=True,
            logging_steps=20, save_strategy="no", report_to="wandb",
            remove_unused_columns=False, optim="paged_adamw_8bit"))
    trainer.train()
    model = model.merge_and_unload(); model.eval()
    wandb.finish()

# ═══ ZERO-SHOT LOGIT SCORING ═══
# Single-token letter IDs for A–E (with and without leading space)
letter_ids = {}
for l in LABELS:
    ids = set()
    for fmt in [l, f" {l}"]:
        enc = tokenizer.encode(fmt, add_special_tokens=False)
        if len(enc) >= 1:
            ids.add(enc[-1])
    letter_ids[l] = list(ids)

SYS = ("You are an expert across science, mathematics, history and philosophy. "
       "Use the provided context if helpful. Respond with ONLY the single "
       "letter of the best answer.")

@torch.inference_mode()
def score_question(row):
    ctx = str(row.get("context", ""))[:1800]
    body = (f"Context: {ctx}\n\n" if ctx else "") + \
           f"Question: {row['prompt']}\n\n" + \
           "\n".join(f"{c}) {row[c]}" for c in LABELS) + \
           "\n\nAnswer with ONLY the correct letter (A, B, C, D, or E):"
    msgs = [{"role": "system", "content": SYS},
            {"role": "user", "content": body}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True)
    inp = tokenizer(text, return_tensors="pt", truncation=True, max_length=2304)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    logits = model(**inp).logits[0, -1].float()
    raw = np.array([max(logits[t].item() for t in letter_ids[l]) for l in LABELS])
    probs = np.exp(raw - raw.max()); probs /= probs.sum()
    return probs

# ── Honest holdout OOF (400 stratified train rows) for ensemble weighting ──
from sklearn.model_selection import train_test_split
hold_idx, _ = train_test_split(np.arange(len(train)), train_size=400,
                               stratify=train["answer"], random_state=42)
hold_idx = np.sort(hold_idx)

safe_wandb_init(project="smart-mcq-solver", name="model3b-qwen-zeroshot-rag",
           config={"model": model_path, "mode": "zero-shot logit scoring",
                   "context": "RAG top-3 paragraphs", "holdout_n": len(hold_idx)})

print(f"\n=== Holdout inference ({len(hold_idx)} train questions) ===")
hp = []
for n, i in enumerate(hold_idx):
    hp.append(score_question(train.iloc[i]))
    if (n + 1) % 100 == 0: print(f"  {n+1}/{len(hold_idx)} done")
hp = np.array(hp)
np.save("llm_holdout_probs.npy", hp)
np.save("llm_holdout_idx.npy", hold_idx)
hold_top3 = [[id2label[j] for j in np.argsort(-p)[:3]] for p in hp]
hold_map3 = map_at_3(train["answer"].values[hold_idx], hold_top3)
print(f"Qwen zero-shot holdout MAP@3: {hold_map3:.4f}")
wandb.log({"holdout_map3": hold_map3})

print(f"\n=== Test Inference ({len(test)} questions) ===")
tp = []
for i, row in test.iterrows():
    tp.append(score_question(row))
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(test)} done")
tp = np.array(tp)
np.save("llm_test_probs.npy", tp)

# Generate LLM-only submission
top3 = [[id2label[j] for j in np.argsort(-p)[:3]] for p in tp]
sub = pd.DataFrame({"ID": test["id"], "Prediction": [" ".join(t) for t in top3]})
sub.to_csv("submission_llm.csv", index=False)
print(f"LLM submission saved: submission_llm.csv")
print(sub.head(10))
wandb.finish()
del model; gc.collect(); torch.cuda.empty_cache()

GPU: Tesla T4
Loading: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

wandb: setting up run zrphs780
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260623_155041-zrphs780
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3b-qwen-zeroshot-rag
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/zrphs780



=== Holdout inference (400 train questions) ===
  100/400 done
  200/400 done
  300/400 done
  400/400 done
Qwen zero-shot holdout MAP@3: 0.8617

=== Test Inference (500 questions) ===
  50/500 done
  100/500 done
  150/500 done
  200/500 done
  250/500 done
  300/500 done
  350/500 done
  400/500 done
  450/500 done


wandb: uploading console lines 17-17; updating run metadata


  500/500 done
LLM submission saved: submission_llm.csv
   ID Prediction
0   1      A E D
1   2      B A D
2   3      C B D
3   4      E A B
4   5      C A B
5   6      A D E
6   7      A E D
7   8      B A D
8   9      A C D
9  10      B C D


wandb: uploading console lines 17-17; uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: uploading console lines 17-17; uploading wandb-summary.json; uploading output.log
wandb: uploading console lines 17-17
wandb: uploading summary, console lines 18-30
wandb: 
wandb: Run history:
wandb: holdout_map3 ▁
wandb: 
wandb: Run summary:
wandb: holdout_map3 0.86167
wandb: 
wandb: 🚀 View run model3b-qwen-zeroshot-rag at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/zrphs780
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260623_155041-zrphs780/logs


In [10]:
import pandas as pd, numpy as np, os

def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']
id2label = {i:l for i,l in enumerate(LABELS)}
y_all = train['answer'].values

# ── Load model predictions ──
model_test, model_oof = {}, {}
for name, oof_f, test_f in [
    ("deberta", "deberta_oof_probs.npy", "deberta_test_probs.npy"),
    ("llm",     "llm_holdout_probs.npy", "llm_test_probs.npy"),
    ("lgbm",    "lgbm_oof_probs.npy",    "lgbm_test_probs.npy"),
    ("textcnn", "textcnn_oof_probs.npy", "textcnn_test_probs.npy"),
]:
    if os.path.exists(test_f):
        model_test[name] = np.load(test_f)
        if os.path.exists(oof_f):
            model_oof[name] = np.load(oof_f)
print(f"Models with test preds: {list(model_test.keys())}")

# ── Tune DeBERTa-vs-Qwen blend weight on the honest holdout ──
W_DEBERTA = 0.65   # fallback if holdout artefacts are missing
if ("deberta" in model_oof and "llm" in model_oof
        and os.path.exists("llm_holdout_idx.npy")):
    hidx = np.load("llm_holdout_idx.npy")
    d_h, q_h = model_oof["deberta"][hidx], model_oof["llm"]
    y_h = y_all[hidx]
    best_w, best_s = 0.65, -1
    for w in np.arange(0.0, 1.01, 0.05):
        blend = w * d_h + (1 - w) * q_h
        top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in blend]
        s = map_at_3(y_h, top3)
        if s > best_s:
            best_s, best_w = s, w
    W_DEBERTA = float(best_w)
    print(f"Holdout-tuned blend: w_deberta={W_DEBERTA:.2f} "
          f"(holdout MAP@3={best_s:.4f})")

# ── Build blended model probabilities ──
if "deberta" in model_test and "llm" in model_test:
    blend_test = W_DEBERTA * model_test["deberta"] + (1 - W_DEBERTA) * model_test["llm"]
elif "deberta" in model_test:
    blend_test = model_test["deberta"]
elif "llm" in model_test:
    blend_test = model_test["llm"]
else:
    # last-resort prior: per-class frequency from train
    prior = train['answer'].value_counts(normalize=True).reindex(LABELS).values
    blend_test = np.tile(prior, (len(test), 1))
    print("WARNING: no GPU model predictions found — falling back to class prior.")

# ── Apply Stage A duplicate override at rank 1 ──
pinned = np.load("dup_pinned.npy") if os.path.exists("dup_pinned.npy") \
         else np.full(len(test), -1)
n_pin = int((pinned >= 0).sum())
print(f"Duplicate-pinned rows: {n_pin}/{len(test)}")

final_top3 = []
for i in range(len(test)):
    order = list(np.argsort(-blend_test[i]))
    if pinned[i] >= 0:
        order = [pinned[i]] + [k for k in order if k != pinned[i]]
    final_top3.append([id2label[k] for k in order[:3]])

sub = pd.DataFrame({"ID": test["id"],
                    "Prediction": [" ".join(t) for t in final_top3]})
sub.to_csv("submission.csv", index=False)

# Diagnostic variant: models only, no duplicate override (LB ablation)
mo_top3 = [[id2label[k] for k in np.argsort(-p)[:3]] for p in blend_test]
pd.DataFrame({"ID": test["id"],
              "Prediction": [" ".join(t) for t in mo_top3]}
             ).to_csv("submission_models_only.csv", index=False)

print(f"\n{'='*50}")
print(f"FINAL submission.csv saved ({len(sub)} rows) — "
      f"dup-pin rank1 on {n_pin} rows, blend w_deberta={W_DEBERTA:.2f}")
print(f"{'='*50}")
print(sub.head(10))

# Validation checks
assert all(len(p.split()) == 3 for p in sub['Prediction']), "Format error: not all rows have 3 predictions!"
assert all(all(x in LABELS for x in p.split()) for p in sub['Prediction']), "Format error: invalid labels!"
assert all(len(set(p.split())) == 3 for p in sub['Prediction']), "Format error: duplicate letters in a row!"
print("\nAll format checks passed!")
print("Ready to submit to Kaggle!")
print("\nTIP: Submit submission.csv first; use submission_models_only.csv and")
print("     submission_dupmatch.csv as LB ablations for the report's error analysis.")

Models with test preds: ['deberta', 'llm', 'lgbm', 'textcnn']
Holdout-tuned blend: w_deberta=0.95 (holdout MAP@3=0.9058)
Duplicate-pinned rows: 475/500

FINAL submission.csv saved (500 rows) — dup-pin rank1 on 475 rows, blend w_deberta=0.95
   ID Prediction
0   1      A E D
1   2      B E A
2   3      B E C
3   4      E C A
4   5      C D A
5   6      D A B
6   7      E C D
7   8      B E C
8   9      C A D
9  10      B E C

All format checks passed!
Ready to submit to Kaggle!

TIP: Submit submission.csv first; use submission_models_only.csv and
     submission_dupmatch.csv as LB ablations for the report's error analysis.
